In [1]:
from datetime import date
import config
import logging
import numpy as np
import os
import pandas as pd

from cryptonalysis.ml_core import transaction_builders, preprocessing, classification
from cryptonalysis.ml_core.market import market

# Logging
logging.basicConfig(level=config.LOGGING_LEVEL)
logger = logging.getLogger()


DATA_FOLDER = 'D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\cryptonalysis\data'
CRYPTO_CURRENCIES = {
    'ETH': "ethereum",
    'BTC': "bitcoin",
    'XRP': "ripple",
    'LTC': "litecoin",
    'USDT': "tether"
}
HISTORICAL_DATA_FILES = {
    key: os.path.join(DATA_FOLDER, "{}_historical.csv".format(value))
    for (key, value) in CRYPTO_CURRENCIES.iteritems()
}

print HISTORICAL_DATA_FILES


{'LTC': 'D:\\Dropbox\\Trabajo\\Cryptocurrencies\\Code\\CryptonalysisServer\\cryptonalysis\\data\\litecoin_historical.csv', 'USDT': 'D:\\Dropbox\\Trabajo\\Cryptocurrencies\\Code\\CryptonalysisServer\\cryptonalysis\\data\\tether_historical.csv', 'ETH': 'D:\\Dropbox\\Trabajo\\Cryptocurrencies\\Code\\CryptonalysisServer\\cryptonalysis\\data\\ethereum_historical.csv', 'XRP': 'D:\\Dropbox\\Trabajo\\Cryptocurrencies\\Code\\CryptonalysisServer\\cryptonalysis\\data\\ripple_historical.csv', 'BTC': 'D:\\Dropbox\\Trabajo\\Cryptocurrencies\\Code\\CryptonalysisServer\\cryptonalysis\\data\\bitcoin_historical.csv'}


In [2]:
# Get runtime parameters
starting_date = date(2017, 1, 1)  # First date for ETH is 2015 8 7
ending_date = date.today()
window_size = 30
lookahead_days = transaction_builders.CryptoPredictor.LOOKAHEAD_DAYS
starting_investment = 100
daily_allowance = 5
prob_buy = 1
prob_sell = 1
crypto_name = 'ETH'
price_column = 'Close'
predictor_params = {
    'prob_buy': 0.8,
    'prob_sell': 0.8,
    'starting_investment': 100,
    'daily_allowance': 5,
    'lookahead_days': 4
}
normalize = True
normalize_by_row = False

# 0. Get DFs
dfs = {
    crypto: preprocessing.get_historical_df(historical_data_file)
    for (crypto, historical_data_file) in HISTORICAL_DATA_FILES.iteritems()
}

if not ending_date:
    ending_date = dfs[crypto_name]['Date'].dt.date.iat[-1]

for crypto, df in dfs.iteritems():
    print "{}:".format(crypto)
    print df.head()


INFO:root:Loading historical data from D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\cryptonalysis\data\litecoin_historical.csv...


INFO:root:Loading historical data from D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\cryptonalysis\data\tether_historical.csv...


INFO:root:Loading historical data from D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\cryptonalysis\data\ethereum_historical.csv...


INFO:root:Loading historical data from D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\cryptonalysis\data\ripple_historical.csv...


INFO:root:Loading historical data from D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\cryptonalysis\data\bitcoin_historical.csv...


LTC:
        Date  Open  High   Low  Close   Volume  Market Cap
0 2015-08-07  4.06  4.22  3.97   4.21  4192810   174598425
1 2015-08-08  4.22  4.22  3.84   3.85  4917730   160042129
2 2015-08-09  3.84  3.98  3.81   3.90  3064680   161969691
3 2015-08-10  3.90  3.98  3.90   3.95  2239890   164169989
4 2015-08-11  3.95  4.16  3.94   4.16  3426300   173045227
USDT:
        Date  Open  High  Low  Close  Volume  Market Cap
0 2015-08-07   1.0   1.0  1.0    1.0  189055      451600
1 2015-08-08   1.0   1.0  1.0    1.0    2229      451600
2 2015-08-09   1.0   1.0  1.0    1.0     657      451600
3 2015-08-10   1.0   1.0  1.0    1.0  152265      451600
4 2015-08-11   1.0   1.0  1.0    1.0     526      451600
ETH:
        Date      Open      High       Low     Close   Volume   Market Cap
0 2015-08-07  2.830000  3.540000  2.520000  2.770000   164329          0.0
1 2015-08-08  2.790000  2.800000  0.714725  0.753325   674188  167911000.0
2 2015-08-09  0.706136  0.879810  0.629191  0.701897   532170  

In [3]:
# 2. Get aggregated DFs
daily_dfs = {}
weekly_dfs = {}
monthly_dfs = {}
for (crypto, df) in dfs.iteritems():
    daily_df, weekly_df, monthly_df = preprocessing.get_aggregated_dfs(df)
    daily_dfs[crypto] = daily_df
    weekly_dfs[crypto] = weekly_df
    monthly_dfs[crypto] = monthly_df
    
daily_dfs[crypto_name].head()

INFO:root:Aggregating data...


INFO:root:Aggregating data...


INFO:root:Aggregating data...


INFO:root:Aggregating data...


INFO:root:Aggregating data...


,Open,High,Low,Close,Volume,Market Cap
Date,,,,,,
2015-08-07,2.830000,3.540000,2.520000,2.770000,164329,0.0
2015-08-08,2.790000,2.800000,0.714725,0.753325,674188,167911000.0
2015-08-09,0.706136,0.879810,0.629191,0.701897,532170,42637600.0
2015-08-10,0.713989,0.729854,0.636546,0.708448,405283,43130000.0
2015-08-11,0.708087,1.130000,0.663235,1.070000,1463100,42796500.0


In [4]:
# 3. Get transaction data
price_lists = {
    crypto: df[price_column]
    for (crypto, df) in daily_dfs.iteritems()
}

price_lists[crypto_name].head()

Date
2015-08-07    2.770000
2015-08-08    0.753325
2015-08-09    0.701897
2015-08-10    0.708448
2015-08-11    1.070000
Name: Close, dtype: float64

In [5]:
# 4. Run transaction builders
transactions_dfs = {}
for (crypto, price_list) in price_lists.iteritems():
    print "*** {} ***".format(crypto)
    predictor = \
        transaction_builders.BiffPredictorSmart(market, starting_date, price_list, window_size, crypto, 
                                                ending_date, starting_investment=starting_investment,
                                                daily_allowance=daily_allowance, lookahead_days=lookahead_days,
                                                prob_buy=prob_buy, prob_sell=prob_sell)
    predictor.run_predictor()
    
    # 5. Get transactions DataFrame
    transactions_df = preprocessing.build_transactions_df(predictor.transactions)
    transactions_dfs[crypto] = transactions_df
    
    logger.info('Buy: {0}'.format(len(transactions_df[transactions_df['transaction'] == 1])))
    logger.info('Sell: {0}'.format(len(transactions_df[transactions_df['transaction'] == 0])))

INFO:root:Running predictor for BiffPredictorSmart...


INFO:root:Predictor parameters:
Starting investment: $100, Daily allowance: $5, Lookahead days: 1


INFO:root:Start date: 2017-01-01


*** LTC ***


INFO:root:Finished running predictor


...

INFO:root:ROI: $2480735.03


INFO:root:********************


INFO:root:Getting transactions DataFrame...


INFO:root:Buy: 466


INFO:root:Sell: 358


In [6]:
print crypto_name
print transactions_dfs[crypto_name].head()

ETH
   price 1  price 2  price 3  price 4  price 5  price 6  price 7  price 8  \
0     8.17     8.38     9.73    11.25    10.25    10.25     9.87    10.29   
1     8.38     9.73    11.25    10.25    10.25     9.87    10.29    10.33   
2     9.73    11.25    10.25    10.25     9.87    10.29    10.33    10.55   
3    11.25    10.25    10.25     9.87    10.29    10.33    10.55     9.72   
4    10.25    10.25     9.87    10.29    10.33    10.55     9.72     9.86   

   price 9  price 10     ...       price 22  price 23  price 24  price 25  \
0    10.33     10.55     ...          10.70     10.82     10.63     10.57   
1    10.55      9.72     ...          10.82     10.63     10.57     10.59   
2     9.72      9.86     ...          10.63     10.57     10.59     10.54   
3     9.86      9.77     ...          10.57     10.59     10.54     10.56   
4     9.77      9.65     ...          10.59     10.54     10.56     10.48   

   price 26  price 27  price 28  price 29  price 30  transaction  
0  

In [7]:
# 6. Data normalization
if normalize:
    for (crypto, transactions_df) in transactions_dfs.iteritems():
        transactions_dfs[crypto] = preprocessing.normalize_df(transactions_df, normalize_by_row)

print transactions_dfs[crypto_name].head()

INFO:root:Normalizing data...


INFO:root:Normalizing data...


INFO:root:Normalizing data...


INFO:root:Normalizing data...


INFO:root:Normalizing data...


          0         1         2         3         4         5         6  \
0 -1.169125 -1.169775 -1.166261 -1.162172 -1.167327 -1.168808 -1.171694   
1 -1.168358 -1.164844 -1.160706 -1.165829 -1.167327 -1.170200 -1.170155   
2 -1.163430 -1.159291 -1.164361 -1.165829 -1.168718 -1.168662 -1.170008   
3 -1.157881 -1.162944 -1.164361 -1.167219 -1.167181 -1.168515 -1.169202   
4 -1.161532 -1.162944 -1.165750 -1.165683 -1.167034 -1.167710 -1.172243   

          7         8         9     ...             21        22        23  \
0 -1.171566 -1.172823 -1.173420     ...      -1.190298 -1.191225 -1.193318   
1 -1.171419 -1.172016 -1.176467     ...      -1.189854 -1.191929 -1.193541   
2 -1.170613 -1.175061 -1.175953     ...      -1.190557 -1.192151 -1.193467   
3 -1.173656 -1.174548 -1.176283     ...      -1.190778 -1.192077 -1.193652   
4 -1.173143 -1.174878 -1.176724     ...      -1.190705 -1.192262 -1.193578   

         24        25        26        27        28        29  transaction  
0 -

In [8]:
# LSTM DL

# import the relevant Keras modules
from keras.models import Sequential
from keras.layers import Activation, Dense
from keras.layers import LSTM
from keras.layers import Dropout


def build_model(inputs, output_size, neurons, activ_func="linear",
                dropout=0.25, loss="mae", optimizer="adam"):
    model = Sequential()

    model.add(LSTM(neurons, input_shape=(inputs.shape[1], inputs.shape[2])))
    model.add(Dropout(dropout))
    model.add(Dense(units=output_size))
    model.add(Activation(activ_func))

    model.compile(loss=loss, optimizer=optimizer)
    return model

Using CNTK backend
D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\venv\lib\site-packages\cntk\cntk_py_init.py:76: UserWarning: 

################################################ Missing optional dependency (    MKL     ) ################################################
   CNTK may crash if the component that depends on those dependencies is loaded.
   Visit https://docs.microsoft.com/en-us/cognitive-toolkit/Setup-Windows-Python#mkl for more information.
############################################################################################################################################

  warnings.warn(WARNING_MSG % ('    MKL     ', 'https://docs.microsoft.com/en-us/cognitive-toolkit/Setup-Windows-Python#mkl'))


D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\venv\lib\site-packages\cntk\cntk_py_init.py:84: UserWarning: 

################################################ Missing optional dependency (GPU-Specific) ################################################
   CNTK may crash if the component that depends on those dependencies is loaded.
   Visit https://docs.microsoft.com/en-us/cognitive-toolkit/Setup-Windows-Python#optional-gpu-specific-packages for more information.
############################################################################################################################################
If you intend to use CNTK without GPU support, you can ignore the (likely) GPU-specific warning!
############################################################################################################################################

  warnings.warn(WARNING_MSG_GPU_ONLY % ('GPU-Specific', 'https://docs.microsoft.com/en-us/cognitive-toolkit/Setup-Windows-Python#optional-gp

D:\Dropbox\Trabajo\Cryptocurrencies\Code\CryptonalysisServer\venv\lib\site-packages\keras\backend\cntk_backend.py:26: UserWarning: CNTK backend warning: GPU is not detected. CNTK's CPU version is not fully optimized,please run with GPU to get better performance.
  'CNTK backend warning: GPU is not detected. '


In [9]:
print transactions_dfs.keys()
print transactions_dfs[crypto_name].shape

['LTC', 'USDT', 'ETH', 'XRP', 'BTC']
(824, 31)


In [10]:
# 1. Split dataset for classification
split_dfs = {
    crypto: {
        'X_dev': {},
        'y_dev': {},
        'X_eval': {},
        'y_eval': {},
    }
    for crypto in transactions_dfs.keys()
}

for (crypto, df) in transactions_dfs.iteritems():
    logger.info("*** {} ***".format(crypto))
    _, _, _, _, _, _, X_dev, y_dev, X_eval, y_eval = \
        classification.split_datasets(df, shuffle=False, training_size=0.7, dev_size=0.5)
    split_dfs[crypto]['X_dev'] = X_dev
    split_dfs[crypto]['y_dev'] = y_dev
    split_dfs[crypto]['X_eval'] = X_eval
    split_dfs[crypto]['y_eval'] = y_eval

INFO:root:*** LTC ***


INFO:root:Length X: 824


INFO:root:Length X_training: 576


INFO:root:Length X_testing: 248


INFO:root:Length X_dev: 412


...

INFO:root:Length X: 824


INFO:root:Length X_training: 576


INFO:root:Length X_testing: 248


INFO:root:Length X_dev: 412


INFO:root:Length X_eval: 412


In [11]:
LSTM_training_inputs = []
LSTM_training_outputs = np.array(split_dfs[crypto_name]['y_dev'])
LSTM_test_inputs = []
LSTM_test_outputs = np.array(split_dfs[crypto_name]['y_eval'])
for (crypto, dfs) in split_dfs.iteritems():
    LSTM_training_inputs.append(dfs['X_dev'])
    LSTM_test_inputs.append(dfs['X_eval'])
    
LSTM_training_inputs = [np.array(LSTM_training_input) for LSTM_training_input in LSTM_training_inputs]
LSTM_training_inputs = np.array(LSTM_training_inputs)
LSTM_training_inputs = np.swapaxes(LSTM_training_inputs, 0, 1)
LSTM_training_inputs = np.swapaxes(LSTM_training_inputs, 1, 2)

LSTM_test_inputs = [np.array(LSTM_test_inputs) for LSTM_test_inputs in LSTM_test_inputs]
LSTM_test_inputs = np.array(LSTM_test_inputs)
LSTM_test_inputs = np.swapaxes(LSTM_test_inputs, 0, 1)
LSTM_test_inputs = np.swapaxes(LSTM_test_inputs, 1, 2)

print LSTM_training_inputs.shape
print LSTM_test_inputs.shape
LSTM_training_inputs[0]

In [15]:
# random seed for reproducibility
np.random.seed(202)

# initialise model architecture
eth_model = build_model(LSTM_training_inputs, output_size=1, neurons=20)

eth_model.get_config()

In [16]:

# train model on data
# note: eth_history contains information on the training error per epoch
eth_history = eth_model.fit(LSTM_training_inputs, LSTM_training_outputs, 
                            epochs=50, batch_size=1, verbose=2, shuffle=True)
